# mid_t Sensitivity Sweep — Lite

Single-checkpoint mid_t sweep for the COMP447 progress report.

- **Checkpoint**: 1980 kimg ECT (`network-snapshot-000198.pkl`)
- **mid_t values**: 8 points across [0.1, 2.5]
- **Samples per point**: 5000 (5k FID screening)
- **Expected runtime**: 15-25 minutes on Colab G4
- **Output**: `project/results/midt_sweep/lite_results.csv` + `lite_curve.png`

## Cell 1: Verify GPU and clone repo

In [9]:
import os, subprocess, torch
print(f"torch: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"gpu: {torch.cuda.get_device_name(0)}")

REPO = "/content/COMP447"
if not os.path.exists(REPO):
    print("Cloning repo...")
    subprocess.check_call(["git", "clone", "https://github.com/bakaraman/COMP447.git", REPO])
else:
    subprocess.check_call(["git", "-C", REPO, "fetch", "origin", "main"])
    subprocess.check_call(["git", "-C", REPO, "reset", "--hard", "origin/main"])
print("Repo synced to:", REPO)

torch: 2.10.0+cu128
cuda available: True
gpu: NVIDIA RTX PRO 6000 Blackwell Server Edition
Repo synced to: /content/COMP447


## Cell 2a: Try to find the checkpoint (Drive or repo)

The 1980 kimg snapshot is 213 MB and gitignored — not in the cloned repo. This cell first tries the standard repo path (in case it was uploaded another way), then mounts Google Drive and globs for the file. If nothing is found, run **Cell 2b** instead and upload directly.

In [4]:
import glob

CHECKPOINT = None

REPO_PATH = f"{REPO}/project/results_backup/ect_checkpoints/network-snapshot-000198.pkl"
if os.path.exists(REPO_PATH):
    CHECKPOINT = REPO_PATH

if CHECKPOINT is None:
    try:
        from google.colab import drive
        if not os.path.exists("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        hits = glob.glob("/content/drive/MyDrive/**/network-snapshot-000198.pkl", recursive=True)
        if hits:
            CHECKPOINT = hits[0]
    except Exception as e:
        print(f"Drive mount skipped: {e}")

for extra in ["/content/network-snapshot-000198.pkl",
              "/content/checkpoint.pkl"]:
    if CHECKPOINT is None and os.path.exists(extra):
        CHECKPOINT = extra

if CHECKPOINT:
    print(f"Checkpoint found: {CHECKPOINT}")
    print(f"Size: {os.path.getsize(CHECKPOINT)/1e6:.1f} MB")
else:
    print("Checkpoint NOT found. Run Cell 2b to upload it directly from your local machine.")

Mounted at /content/drive
Checkpoint found: /content/drive/MyDrive/COMP447_checkpoints/network-snapshot-000198.pkl
Size: 223.2 MB


## Cell 2b: Upload the checkpoint directly (fallback)

Only run this if Cell 2a did not find the checkpoint. Opens a file picker; choose `network-snapshot-000198.pkl` from `/Users/batuhankaraman/Downloads/COMP447/project/results_backup/ect_checkpoints/` on your laptop. Upload takes 1-3 minutes for 213 MB. After this completes, re-run Cell 2a to verify the path is now set.

In [5]:
from google.colab import files
uploaded = files.upload()
for fname in uploaded.keys():
    src = f"/content/{fname}"
    if os.path.exists(src):
        CHECKPOINT = src
        print(f"Uploaded: {CHECKPOINT}  ({os.path.getsize(CHECKPOINT)/1e6:.1f} MB)")
        break

KeyboardInterrupt: 

## Cell 3: Setup the ECT environment (one-time)

ECT brings pinned versions and a small monkey-patch for PyTorch 2.x. Idempotent — safe to re-run.

In [7]:
setup_log = subprocess.run(
    ["bash", f"{REPO}/project/scripts/setup_ect.sh"],
    capture_output=True, text=True, cwd=f"{REPO}/project",
)
print(setup_log.stdout[-2000:])
if setup_log.returncode != 0:
    print("STDERR:", setup_log.stderr[-2000:])
    raise RuntimeError("setup_ect.sh failed")

Repo root: /content/COMP447

Cloning or refreshing ECT...
  patched: InfiniteSampler.__init__ for torch >= 2.2

Cloning or refreshing EDM...

Bootstrap complete.

Next steps:
  1. Create the runtime environment shown in:
     - /content/COMP447/project/src/ect/env.yml
     - /content/COMP447/project/src/edm/environment.yml

  2. Prepare CIFAR-10 in EDM format:
     cd /content/COMP447/project/src/ect
     python3 dataset_tool.py --source=/path/to/cifar-10-python.tar.gz --dest=datasets/cifar10-32x32.zip

  3. Run an ECT sanity check:
     cd /content/COMP447/project/src/ect
     bash run_ecm_1hour.sh 1 29500 --desc monday_sanity

  4. Evaluate an ECT checkpoint:
     cd /content/COMP447/project/src/ect
     bash eval_ecm.sh 1 29501 --resume /path/to/network-snapshot.pkl

  5. Measure latency locally once checkpoints are available:
     cd /content/COMP447
     python3 project/scripts/measure_latency.py --checkpoint /path/to/checkpoint.pkl --sampler ect --steps 2 --batch_size 1

  6. Heu

## Cell 4: Run the lite sweep

8 mid_t values × 5000 samples × FID via EDM `fid.py`. Streams progress to the cell output. CSV is rewritten after each completed point so partial progress is recoverable if the kernel disconnects.

In [10]:
import sys
assert CHECKPOINT and os.path.exists(CHECKPOINT), "CHECKPOINT not set — run Cell 2a or 2b first."
cmd = [
    "python3", f"{REPO}/project/scripts/midt_sweep_lite.py",
    "--checkpoint", CHECKPOINT,
    "--num", "5000",
    "--gen_batch", "64",
    "--fid_batch", "64",
    "--output_dir", "project/results/midt_sweep",
    "--repo_root", REPO,
]
print("$", " ".join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
proc.wait()
print("\nreturn code:", proc.returncode)

$ python3 /content/COMP447/project/scripts/midt_sweep_lite.py --checkpoint /content/drive/MyDrive/COMP447_checkpoints/network-snapshot-000198.pkl --num 5000 --gen_batch 64 --fid_batch 64 --output_dir project/results/midt_sweep --repo_root /content/COMP447
[mid_t=0.100] running: python3 /content/COMP447/project/scripts/eval_fid.py ect --checkpoint /content/drive/MyDrive/COMP447_checkpoints/network-snapshot-000198.pkl --steps 2 --mid_t 0.1 --num 5000 --gen_batch 64 --fid_batch 64 --seed 0
/content/COMP447/project/scripts/eval_fid.py:69: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(img, "RGB").save(outdir / f"{start_idx + i:06d}.png")
Generating ECT images: steps=2, mid_t=0.1, num=5000, batch=64, checkpoint=/content/drive/MyDrive/COMP447_checkpoints/network-snapshot-000198.pkl
  saved 1024/5000 images
  saved 2048/5000 images
  saved 3072/5000 images
  saved 4096/5000 images
  saved 5000/5000 images
$ torchrun --standal

## Cell 5: Find all 4 checkpoints in Drive

We need 500 / 1000 / 1500 / 1980 kimg snapshots (`network-snapshot-{050,100,150,198}.pkl`). The 1980 sweep already ran above; this cell verifies the others exist before kicking off the multi-checkpoint sweep.

In [ ]:
import glob, os
ALL_CKPTS = sorted(glob.glob('/content/drive/MyDrive/COMP447_checkpoints/network-snapshot-*.pkl'))
for p in ALL_CKPTS:
    print(f'{p}  ({os.path.getsize(p)/1e6:.1f} MB)')
assert len(ALL_CKPTS) >= 4, f'Expected 4 checkpoints in Drive, found {len(ALL_CKPTS)}'
print(f'\n{len(ALL_CKPTS)} checkpoints ready.')

## Cell 6: Multi-checkpoint sweep

Runs the same 8 mid_t values on the other 3 checkpoints (500, 1000, 1500 kimg). Reuses the existing 1980 kimg CSV. Per-checkpoint output goes to `sweep_<snap_id>.csv` so partial progress is preserved if the kernel disconnects.

Expected runtime: ~12-15 min (3 ckpts × 8 mid_t × ~30s).

In [ ]:
import shutil, sys, subprocess
results_root = f'{REPO}/project/results/midt_sweep'
os.makedirs(results_root, exist_ok=True)

# Stash the existing 1980 sweep CSV under a kimg-tagged name (idempotent).
existing_198 = f'{results_root}/lite_results.csv'
target_198 = f'{results_root}/sweep_198.csv'
if os.path.exists(existing_198) and not os.path.exists(target_198):
    shutil.copy(existing_198, target_198)
    print(f'Stashed existing 1980 results to {target_198}')

tmp_dir = f'{REPO}/project/results/midt_sweep_tmp'
for ckpt_path in ALL_CKPTS:
    fname = os.path.basename(ckpt_path)
    snap_id = fname.split('-')[-1].replace('.pkl','')[-3:]  # e.g. '050', '198'
    out_csv = f'{results_root}/sweep_{snap_id}.csv'
    if os.path.exists(out_csv):
        print(f'[{snap_id}] already done, skipping')
        continue
    print(f'\n=== [{snap_id}] starting sweep on {ckpt_path} ===')
    cmd = [
        'python3', f'{REPO}/project/scripts/midt_sweep_lite.py',
        '--checkpoint', ckpt_path,
        '--num', '5000',
        '--gen_batch', '64',
        '--fid_batch', '64',
        '--output_dir', 'project/results/midt_sweep_tmp',
        '--repo_root', REPO,
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        sys.stdout.write(line); sys.stdout.flush()
    proc.wait()
    if proc.returncode != 0:
        print(f'[{snap_id}] FAILED rc={proc.returncode}')
        break
    tmp_csv = f'{tmp_dir}/lite_results.csv'
    shutil.copy(tmp_csv, out_csv)
    print(f'[{snap_id}] saved to {out_csv}')
print('\n--- multi-ckpt sweep done ---')

## Cell 7: Merge per-checkpoint CSVs and inspect

Combines the four per-checkpoint files into a single `multi_checkpoint.csv` and prints best mid_t per checkpoint.

In [ ]:
import pandas as pd
import glob as glob_mod
results_root = f'{REPO}/project/results/midt_sweep'
files = sorted(glob_mod.glob(f'{results_root}/sweep_*.csv'))
all_dfs = []
for f in files:
    snap_id = os.path.basename(f).split('_')[1].replace('.csv','')
    kimg = int(snap_id) * 10
    df = pd.read_csv(f)
    df['kimg'] = kimg
    df['snap_id'] = snap_id
    all_dfs.append(df)
multi = pd.concat(all_dfs, ignore_index=True)
multi = multi[['kimg','snap_id','mid_t','fid','n_samples','wall_s']].sort_values(['kimg','mid_t'])
multi.to_csv(f'{results_root}/multi_checkpoint.csv', index=False)
print(multi.to_string(index=False))
print('\n--- Best mid_t per checkpoint ---')
for kimg, g in multi.groupby('kimg'):
    best = g.sort_values('fid').iloc[0]
    default_row = g[g['mid_t']==0.821]
    default_fid = float(default_row['fid'].iloc[0]) if len(default_row) else float('nan')
    print(f'  {kimg} kimg: best mid_t={float(best["mid_t"]):.3f} FID={float(best["fid"]):.4f}  vs default 0.821 FID={default_fid:.4f}  delta={default_fid - float(best["fid"]):.4f}')

## Cell 8: Plot all 4 checkpoint curves together

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display
fig, ax = plt.subplots(figsize=(6.4, 4.0))
palette = {500:'#fee5d9', 1000:'#fcae91', 1500:'#fb6a4a', 1980:'#a83d3d'}
for kimg, g in multi.groupby('kimg'):
    g = g.sort_values('mid_t')
    color = palette.get(int(kimg), 'gray')
    ax.plot(g['mid_t'], g['fid'], 'o-', color=color, lw=2, ms=6, label=f'{int(kimg)} kimg')
    best = g.sort_values('fid').iloc[0]
    ax.scatter([best['mid_t']], [best['fid']], color=color, edgecolor='black', marker='*', s=180, zorder=5)
ax.axvline(0.821, color='gray', ls='--', lw=1, label='ECT default 0.821')
ax.set_xlabel(r'$t_{\mathrm{mid}}$')
ax.set_ylabel('FID (5k samples)')
ax.set_title('mid_t sweep across ECT checkpoints · CIFAR-10')
ax.grid(alpha=0.3)
ax.legend(loc='best', fontsize=9)
fig.tight_layout()
plot_path = f'{results_root}/multi_checkpoint.png'
fig.savefig(plot_path, dpi=140)
print(f'Saved: {plot_path}')
display(Image(plot_path))

## Cell 9: Push results to GitHub

Pushes the multi-checkpoint CSV and PNG so the local repo can pull them and update the report.

In [ ]:
rs = lambda cmd: subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
print(rs(['git', 'add', 'project/results/midt_sweep']).stdout)
print(rs(['git', '-c', 'user.email=colab@batuhan', '-c', 'user.name=colab',
         'commit', '-m', 'Add multi-checkpoint mid_t sweep results']).stdout)
out = rs(['git', 'push', 'origin', 'main']).stdout
print(out if out else 'pushed')
print('--- now run locally: git pull origin main ---')